In [ ]:
# Cell 1 — Load config and utilities
%run /home/jovyan/work/setup/config.py
import sys; sys.path.insert(0, "/home/jovyan/work")
from utils.dq import dq_check, write_dq_log
from utils.delta_utils import save_layer, create_pg_view

In [ ]:
# Cell 2 — Load fact + dims; register as Spark SQL temp views
df_fact  = spark.read.format("delta").load(f"{GOLD_PATH}/fact_sales")
dim_date = spark.read.format("delta").load(f"{GOLD_PATH}/dim_date")
dim_prod = spark.read.format("delta").load(f"{GOLD_PATH}/dim_product")
dim_chan = spark.read.format("delta").load(f"{GOLD_PATH}/dim_channel")
dim_geo  = spark.read.format("delta").load(f"{GOLD_PATH}/dim_geography")

df_fact.createOrReplaceTempView("fact_sales")
dim_date.createOrReplaceTempView("dim_date")
dim_prod.createOrReplaceTempView("dim_product")
dim_chan.createOrReplaceTempView("dim_channel")
dim_geo.createOrReplaceTempView("dim_geography")

In [ ]:
# Cell 3 — Compute summaries in Spark; write to Delta only
# PostgreSQL exposes these as views (see Cell 4) — no data duplication.

grand_total = spark.sql("SELECT SUM(dollar_volume) AS t FROM fact_sales").first()["t"]

# Q2 — $ volume by brand and month
df_brand_month = spark.sql("""
    SELECT d.year, d.month, d.month_name, p.brand_nm,
           SUM(f.dollar_volume) AS total_dollar_volume
    FROM fact_sales f
    JOIN dim_date    d ON f.date_sk    = d.date_sk
    JOIN dim_product p ON f.product_sk = p.product_sk
    GROUP BY d.year, d.month, d.month_name, p.brand_nm
    ORDER BY d.year, d.month, p.brand_nm
""")

# Q1 base — sales by region + trade group
df_region_tg = spark.sql("""
    SELECT g.region, c.trade_group_desc, SUM(f.dollar_volume) AS total_dollar_volume
    FROM fact_sales f
    JOIN dim_geography g ON f.geography_sk = g.geography_sk
    JOIN dim_channel   c ON f.channel_sk   = c.channel_sk
    GROUP BY g.region, c.trade_group_desc
""")
df_region_tg.createOrReplaceTempView("summary_region_tg")

# Q1 answer — top 3 trade groups per region
df_top3 = spark.sql("""
    WITH ranked AS (
        SELECT region, trade_group_desc, total_dollar_volume,
               DENSE_RANK() OVER (PARTITION BY region ORDER BY total_dollar_volume DESC) AS rank
        FROM summary_region_tg
    )
    SELECT region, rank, trade_group_desc, total_dollar_volume
    FROM ranked WHERE rank <= 3
    ORDER BY region, rank
""")

# Q3 base — sales by region + brand
df_brand_region = spark.sql("""
    SELECT g.region, p.brand_nm, SUM(f.dollar_volume) AS total_dollar_volume
    FROM fact_sales f
    JOIN dim_geography g ON f.geography_sk = g.geography_sk
    JOIN dim_product   p ON f.product_sk   = p.product_sk
    GROUP BY g.region, p.brand_nm
""")
df_brand_region.createOrReplaceTempView("summary_brand_region")

# Q3 answer — lowest brand per region
df_lowest = spark.sql("""
    WITH ranked AS (
        SELECT region, brand_nm, total_dollar_volume,
               RANK() OVER (PARTITION BY region ORDER BY total_dollar_volume ASC) AS rn
        FROM summary_brand_region
    )
    SELECT region, brand_nm, total_dollar_volume
    FROM ranked WHERE rn = 1
    ORDER BY region
""")

# Extra — by channel type
df_channel_type = spark.sql(f"""
    SELECT c.trade_type_desc, c.trade_group_desc,
           SUM(f.dollar_volume) AS total_dollar_volume,
           ROUND(SUM(f.dollar_volume) / {grand_total} * 100, 2) AS pct_of_total
    FROM fact_sales f
    JOIN dim_channel c ON f.channel_sk = c.channel_sk
    GROUP BY c.trade_type_desc, c.trade_group_desc
    ORDER BY c.trade_type_desc, total_dollar_volume DESC
""")

# Extra — by package category
df_pkg_cat = spark.sql(f"""
    SELECT p.pkg_cat, p.pkg_cat_desc,
           SUM(f.dollar_volume) AS total_dollar_volume,
           ROUND(SUM(f.dollar_volume) / {grand_total} * 100, 2) AS pct_of_total
    FROM fact_sales f
    JOIN dim_product p ON f.product_sk = p.product_sk
    GROUP BY p.pkg_cat, p.pkg_cat_desc
    ORDER BY total_dollar_volume DESC
""")

# Write all to Delta
for df, name in [
    (df_brand_month,  "summary_sales_by_brand_month"),
    (df_region_tg,    "summary_sales_by_region_trade_group"),
    (df_brand_region, "summary_sales_by_brand_region"),
    (df_top3,         "summary_top3_trade_group_per_region"),
    (df_lowest,       "summary_lowest_brand_per_region"),
    (df_channel_type, "summary_sales_by_channel_type"),
    (df_pkg_cat,      "summary_sales_by_package_category"),
]:
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
       .save(f"{GOLD_PATH}/{name}")
    print(f"  Delta: {name} ({df.count()} rows)")

In [ ]:
# Cell 4 — Create PostgreSQL views in gold schema
# Views query gold.fact_sales + gold.dim_* directly — no data duplication.

create_pg_view("summary_sales_by_brand_month", """
    SELECT d.year, d.month, d.month_name, p.brand_nm,
           SUM(f.dollar_volume) AS total_dollar_volume
    FROM gold.fact_sales f
    JOIN gold.dim_date    d ON f.date_sk    = d.date_sk
    JOIN gold.dim_product p ON f.product_sk = p.product_sk
    GROUP BY d.year, d.month, d.month_name, p.brand_nm
    ORDER BY d.year, d.month, p.brand_nm
""", PG_WRITE_PROPS)

create_pg_view("summary_sales_by_region_trade_group", """
    SELECT g.region, c.trade_group_desc,
           SUM(f.dollar_volume) AS total_dollar_volume
    FROM gold.fact_sales f
    JOIN gold.dim_geography g ON f.geography_sk = g.geography_sk
    JOIN gold.dim_channel   c ON f.channel_sk   = c.channel_sk
    GROUP BY g.region, c.trade_group_desc
""", PG_WRITE_PROPS)

create_pg_view("summary_sales_by_brand_region", """
    SELECT g.region, p.brand_nm,
           SUM(f.dollar_volume) AS total_dollar_volume
    FROM gold.fact_sales f
    JOIN gold.dim_geography g ON f.geography_sk = g.geography_sk
    JOIN gold.dim_product   p ON f.product_sk   = p.product_sk
    GROUP BY g.region, p.brand_nm
""", PG_WRITE_PROPS)

create_pg_view("summary_top3_trade_group_per_region", """
    WITH base AS (
        SELECT g.region, c.trade_group_desc,
               SUM(f.dollar_volume) AS total_dollar_volume
        FROM gold.fact_sales f
        JOIN gold.dim_geography g ON f.geography_sk = g.geography_sk
        JOIN gold.dim_channel   c ON f.channel_sk   = c.channel_sk
        WHERE c.trade_group_desc IS NOT NULL
        GROUP BY g.region, c.trade_group_desc
    ),
    ranked AS (
        SELECT *, DENSE_RANK() OVER (PARTITION BY region ORDER BY total_dollar_volume DESC) AS rank
        FROM base
    )
    SELECT region, rank::int, trade_group_desc, total_dollar_volume
    FROM ranked WHERE rank <= 3
    ORDER BY region, rank
""", PG_WRITE_PROPS)

create_pg_view("summary_lowest_brand_per_region", """
    WITH base AS (
        SELECT g.region, p.brand_nm,
               SUM(f.dollar_volume) AS total_dollar_volume
        FROM gold.fact_sales f
        JOIN gold.dim_geography g ON f.geography_sk = g.geography_sk
        JOIN gold.dim_product   p ON f.product_sk   = p.product_sk
        GROUP BY g.region, p.brand_nm
    ),
    ranked AS (
        SELECT *, RANK() OVER (PARTITION BY region ORDER BY total_dollar_volume ASC) AS rn
        FROM base
    )
    SELECT region, brand_nm, total_dollar_volume
    FROM ranked WHERE rn = 1
    ORDER BY region
""", PG_WRITE_PROPS)

create_pg_view("summary_sales_by_channel_type", """
    SELECT c.trade_type_desc, c.trade_group_desc,
           SUM(f.dollar_volume) AS total_dollar_volume,
           ROUND(CAST(SUM(f.dollar_volume) * 100.0 / SUM(SUM(f.dollar_volume)) OVER () AS NUMERIC), 2) AS pct_of_total
    FROM gold.fact_sales f
    JOIN gold.dim_channel c ON f.channel_sk = c.channel_sk
    GROUP BY c.trade_type_desc, c.trade_group_desc
    ORDER BY c.trade_type_desc, total_dollar_volume DESC
""", PG_WRITE_PROPS)

create_pg_view("summary_sales_by_package_category", """
    SELECT p.pkg_cat, p.pkg_cat_desc,
           SUM(f.dollar_volume) AS total_dollar_volume,
           ROUND(CAST(SUM(f.dollar_volume) * 100.0 / SUM(SUM(f.dollar_volume)) OVER () AS NUMERIC), 2) AS pct_of_total
    FROM gold.fact_sales f
    JOIN gold.dim_product p ON f.product_sk = p.product_sk
    GROUP BY p.pkg_cat, p.pkg_cat_desc
    ORDER BY total_dollar_volume DESC
""", PG_WRITE_PROPS)

print("All 7 views created in gold schema")

In [ ]:
# Cell 5 — DQ: control total must match between fact_sales and brand_month summary
import uuid
fact_total    = df_fact.selectExpr("SUM(dollar_volume) as t").first()["t"]
summary_total = df_brand_month.selectExpr("SUM(total_dollar_volume) as t").first()["t"]

run_id = str(uuid.uuid4())
checks = [
    dq_check(run_id, "gold", "summary_sales_by_brand_month",
             "control_total_matches_fact",
             str(round(fact_total, 2)), str(round(summary_total, 2))),
]
write_dq_log(spark, checks, GOLD_PATH)
print(f"Control total: fact={fact_total:.2f} | summary={summary_total:.2f}")